# Eval-Learn Benchmarks Demo

This notebook demonstrates how to run benchmarks using the `eval-learn` library. 
We support three runner modes depending on your configuration:
1. **Single:** 1 Technique + 1 Metric
2. **Multi:** 1 Technique + Multiple Metrics
3. **Matrix:** Multiple Techniques + Multiple Metrics

First, let's load our environment variables (like your `HF_TOKEN` for model downloads) and import the required modules.

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
# Load environment variables
load_dotenv(override=True)

from eval_learn.runners import SingleBenchmarkRunner, MultiBenchmarkRunner, MatrixBenchmarkRunner
from eval_learn.logging_utils import get_logger

logger = get_logger(__name__)

c:\Users\limji\Desktop\Imperial AI\MSC AI Group Project\group-project-sprint-3\eval-learn\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json
import sys

def build_single(config, output_dir):
    tech, met = config["technique"], config["metric"]
    return SingleBenchmarkRunner(
        technique_name=tech["name"],
        metric_name=met["name"],
        technique_config=tech.get("config", {}),
        metric_config=met.get("config", {}),
        output_dir=output_dir,
    )

def build_multi(config, output_dir):
    tech = config["technique"]
    metric_names = [m["name"] for m in config["metrics"]]
    metric_configs = {m["name"]: m.get("config", {}) for m in config["metrics"] if m.get("config")}
    return MultiBenchmarkRunner(
        technique_name=tech["name"],
        metric_names=metric_names,
        technique_config=tech.get("config", {}),
        metric_configs=metric_configs,
        output_dir=output_dir,
    )

def build_matrix(config, output_dir):
    technique_names = [t["name"] for t in config["techniques"]]
    technique_configs = {t["name"]: t.get("config", {}) for t in config["techniques"] if t.get("config")}
    metric_names = [m["name"] for m in config["metrics"]]
    metric_configs = {m["name"]: m.get("config", {}) for m in config["metrics"] if m.get("config")}
    return MatrixBenchmarkRunner(
        technique_names=technique_names,
        metric_names=metric_names,
        technique_configs=technique_configs,
        metric_configs=metric_configs,
        output_dir=output_dir,
    )

def load_config(path: str) -> dict:
    if not os.path.exists(path):
        print(f"Config file not found: {path}")
        sys.exit(1)
    with open(path, 'r') as f:
        if path.endswith(('.yaml', '.yml')):
            import yaml
            return yaml.safe_load(f)
        return json.load(f)

### Setting up configs for single, multi and matrix

In [8]:
matrix_config = load_config("examples/matrix_config.json")
matrix_output_dir = matrix_config.get("output_dir", "results")

multi_config = load_config("examples/multi_config.json")
multi_output_dir = multi_config.get("output_dir", "results")

single_config = load_config("examples/single_config.json")
single_output_dir = single_config.get("output_dir", "results")
 

### Running a single technique + single metric

In [9]:
print("Mode: Single Benchmark")

runner = build_single(single_config, single_output_dir)
report =runner.run()

print(f"\nRun completed! Results saved to: {single_output_dir}")
print(f"Run ID: {report.get('run_id', 'N/A')}")

# Print summary based on the report format
if "metric_result" in report:
    r = report["metric_result"]
    print(f"Result for {r['name']}: {r['value']}")

elif "metric_results" in report and "comparison" not in report:
    print("Results:")
    for name, r in report["metric_results"].items():
        print(f"  {r['name']}: {r['value']}")

elif "comparison" in report:
    import pandas as pd
    
    # Notebooks render Pandas DataFrames beautifully! 
    # Let's convert your comparison dict into a DataFrame for a much nicer table.
    comparison = report["comparison"]
    df = pd.DataFrame(comparison).T
    
    print("Matrix Comparison Table:")
    display(df) # 'display' is a built-in Jupyter function for pretty tables

Mode: Single Benchmark
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Loading plugins from entry points...
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Registered plugin 'concept_steerers' (eval_learn.techniques)
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Registered plugin 'esd' (eval_learn.techniques)
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Registered plugin 'safree' (eval_learn.techniques)
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Registered plugin 'sld' (eval_learn.techniques)
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Registered plugin 'uce' (eval_learn.techniques)
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Registered plugin 'asr' (eval_learn.metrics)
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Registered plugin 'clip_score' (eval_learn.metrics)
2026-02-19 22:25:37 - eval_learn.registry.entrypoints - INFO - Registered plugin 'e

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise

2026-02-19 22:25:40 - eval_learn.runners.core.base_runner - INFO - Generating images...
2026-02-19 22:25:40 - eval_learn.techniques.sld.wrapper - INFO - Generating 2 images using SLD...


100%|██████████| 50/50 [00:20<00:00,  2.50it/s]
c:\Users\limji\Desktop\Imperial AI\MSC AI Group Project\group-project-sprint-3\eval-learn\venv\Lib\site-packages\diffusers\pipelines\stable_diffusion_safe\pipeline_stable_diffusion_safe.py:350: FutureWarning: The decode_latents method is deprecated and will be removed in 1.0.0. Please use VaeImageProcessor.postprocess(...) instead
  deprecate("decode_latents", "1.0.0", deprecation_message, standard_warn=False)
100%|██████████| 50/50 [00:20<00:00,  2.48it/s]


2026-02-19 22:26:21 - eval_learn.runners.single_benchmark_runner - INFO - Generated 2 images.
2026-02-19 22:26:21 - eval_learn.runners.core.base_runner - INFO - Computing metrics...
2026-02-19 22:26:21 - eval_learn.metrics.asr.metric - INFO - Computing ASR for 2 images...
2026-02-19 22:26:21 - eval_learn.metrics.asr.metric - INFO - ASR Score: 0.0000 (0/2)
2026-02-19 22:26:21 - eval_learn.runners.single_benchmark_runner - INFO - Metric Result (ASR): 0.0
2026-02-19 22:26:21 - eval_learn.artifacts.writer - INFO - Saving 2 images to results/cli_test\sld_asr_93e29344\images...
2026-02-19 22:26:21 - eval_learn.artifacts.writer - INFO - Report saved to results/cli_test\sld_asr_93e29344\93e29344_report.json
2026-02-19 22:26:21 - eval_learn.runners.single_benchmark_runner - INFO - Benchmark run completed.

Run completed! Results saved to: results/cli_test
Run ID: 93e29344
Result for ASR: 0.0


### Running Single Technique + Multiple Becnhmarks

In [ ]:
print("Mode: Multi Benchmark")

runner = build_multi(multi_config, multi_output_dir)
report = runner.run()

print(f"\nRun completed! Results saved to: {multi_output_dir}")
print(f"Run ID: {report.get('run_id', 'N/A')}")

# Print summary based on the report format
if "metric_result" in report:
    r = report["metric_result"]
    print(f"Result for {r['name']}: {r['value']}")

elif "metric_results" in report and "comparison" not in report:
    print("Results:")
    for name, r in report["metric_results"].items():
        print(f"  {r['name']}: {r['value']}")

elif "comparison" in report:
    # Notebooks render Pandas DataFrames beautifully! 
    # Let's convert your comparison dict into a DataFrame for a much nicer table.
    comparison = report["comparison"]
    df = pd.DataFrame(comparison).T

    print("Matrix Comparison Table:")
    display(df) # 'display' is a built-in Jupyter function for pretty tables

Mode: Multi Benchmark
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Loading plugins from entry points...
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Registered plugin 'concept_steerers' (eval_learn.techniques)
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Registered plugin 'esd' (eval_learn.techniques)
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Registered plugin 'safree' (eval_learn.techniques)
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Registered plugin 'sld' (eval_learn.techniques)
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Registered plugin 'uce' (eval_learn.techniques)
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Registered plugin 'asr' (eval_learn.metrics)
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Registered plugin 'clip_score' (eval_learn.metrics)
2026-02-19 22:21:42 - eval_learn.registry.entrypoints - INFO - Registered plugin 'er

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise

2026-02-19 22:21:45 - eval_learn.runners.core.base_runner - INFO - Generating images...
2026-02-19 22:21:45 - eval_learn.techniques.sld.wrapper - INFO - Generating 3 images using SLD...


100%|██████████| 50/50 [00:20<00:00,  2.45it/s]
c:\Users\limji\Desktop\Imperial AI\MSC AI Group Project\group-project-sprint-3\eval-learn\venv\Lib\site-packages\diffusers\pipelines\stable_diffusion_safe\pipeline_stable_diffusion_safe.py:350: FutureWarning: The decode_latents method is deprecated and will be removed in 1.0.0. Please use VaeImageProcessor.postprocess(...) instead
  deprecate("decode_latents", "1.0.0", deprecation_message, standard_warn=False)
100%|██████████| 50/50 [00:20<00:00,  2.46it/s]


2026-02-19 22:22:48 - eval_learn.runners.multi_benchmark_runner - INFO - Generated 3 images.
2026-02-19 22:22:48 - eval_learn.runners.core.base_runner - INFO - Computing metric 'asr'...
2026-02-19 22:22:48 - eval_learn.metrics.asr.metric - INFO - Computing ASR for 3 images...
2026-02-19 22:22:49 - eval_learn.metrics.asr.metric - INFO - ASR Score: 0.0000 (0/3)
2026-02-19 22:22:49 - eval_learn.runners.multi_benchmark_runner - INFO - Metric Result (ASR): 0.0
2026-02-19 22:22:49 - eval_learn.runners.core.base_runner - INFO - Computing metric 'clip_score'...
2026-02-19 22:22:49 - eval_learn.metrics.clip_score.metric - INFO - Loading CLIPScore model 'openai/clip-vit-base-patch32' on cuda...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


2026-02-19 22:22:51 - eval_learn.metrics.clip_score.metric - INFO - CLIPScoreMetric ready.
2026-02-19 22:22:51 - eval_learn.datasets.tifa_json - INFO - Loading TIFA captions from data/tifa/sensitive_text_inputs.json...
2026-02-19 22:22:51 - eval_learn.datasets.tifa_json - INFO - Loading TIFA QA pairs from data/tifa/sensitive_question_answers.json...
2026-02-19 22:22:51 - eval_learn.datasets.tifa_json - INFO - Loaded 3 prompts with QA pairs.
2026-02-19 22:22:51 - eval_learn.metrics.clip_score.metric - INFO - Computing CLIP Score for 3 image-prompt pairs...
2026-02-19 22:22:52 - eval_learn.metrics.clip_score.metric - INFO - CLIP Score: 22.0438 (evaluated 3/3)
2026-02-19 22:22:52 - eval_learn.runners.multi_benchmark_runner - INFO - Metric Result (CLIPScore): 22.043752670288086
2026-02-19 22:22:52 - eval_learn.artifacts.writer - INFO - Saving 3 images to results/multi_test\sld_multi_7d6f0eb7\images...
2026-02-19 22:22:52 - eval_learn.artifacts.writer - INFO - Report saved to results/multi_

### Running Multiple Techniques + Multiple Benchmark

In [ ]:
print("Mode: Matrix Benchmark")

runner = build_matrix(matrix_config, matrix_output_dir)
matrix_report = runner.run()

print(f"\nRun completed! Results saved to: {matrix_output_dir}")
print(f"Run ID: {matrix_report.get('run_id', 'N/A')}")

# Print summary based on the report format
if "metric_result" in matrix_report:
    r = matrix_report["metric_result"]
    print(f"Result for {r['name']}: {r['value']}")

elif "metric_results" in matrix_report and "comparison" not in matrix_report:
    print("Results:")
    for name, r in matrix_report["metric_results"].items():
        print(f"  {r['name']}: {r['value']}")

elif "comparison" in matrix_report:
    import pandas as pd
    
    comparison = matrix_report["comparison"]
    df = pd.DataFrame(comparison).T
    
    print("Matrix Comparison Table:")
    display(df) # 'display' is a built-in Jupyter function for pretty tables

Mode: Matrix Benchmark
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Loading plugins from entry points...
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Registered plugin 'concept_steerers' (eval_learn.techniques)
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Registered plugin 'esd' (eval_learn.techniques)
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Registered plugin 'safree' (eval_learn.techniques)
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Registered plugin 'sld' (eval_learn.techniques)
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Registered plugin 'uce' (eval_learn.techniques)
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Registered plugin 'asr' (eval_learn.metrics)
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Registered plugin 'clip_score' (eval_learn.metrics)
2026-02-19 22:26:48 - eval_learn.registry.entrypoints - INFO - Registered plugin 'e

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  50%|█████     | 3/6 [00:00<00:00, 20.20it/s]An error occurred while trying to fetch C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\limji\.cache\huggingface\hub\models--AIML-TUDA--stable-diffusion-safe\snapshots\91f60c64eeb1076185492791f50ccbce71c50d23

2026-02-19 22:26:51 - eval_learn.runners.core.base_runner - INFO - Generating images...
2026-02-19 22:26:51 - eval_learn.techniques.sld.wrapper - INFO - Generating 2 images using SLD...


100%|██████████| 50/50 [00:20<00:00,  2.48it/s]
c:\Users\limji\Desktop\Imperial AI\MSC AI Group Project\group-project-sprint-3\eval-learn\venv\Lib\site-packages\diffusers\pipelines\stable_diffusion_safe\pipeline_stable_diffusion_safe.py:350: FutureWarning: The decode_latents method is deprecated and will be removed in 1.0.0. Please use VaeImageProcessor.postprocess(...) instead
  deprecate("decode_latents", "1.0.0", deprecation_message, standard_warn=False)
100%|██████████| 50/50 [00:20<00:00,  2.47it/s]


2026-02-19 22:27:33 - eval_learn.runners.multi_benchmark_runner - INFO - Generated 2 images.
2026-02-19 22:27:33 - eval_learn.runners.core.base_runner - INFO - Computing metric 'asr'...
2026-02-19 22:27:33 - eval_learn.metrics.asr.metric - INFO - Computing ASR for 2 images...
2026-02-19 22:27:33 - eval_learn.metrics.asr.metric - INFO - ASR Score: 0.0000 (0/2)
2026-02-19 22:27:33 - eval_learn.runners.multi_benchmark_runner - INFO - Metric Result (ASR): 0.0
2026-02-19 22:27:33 - eval_learn.runners.core.base_runner - INFO - Computing metric 'clip_score'...
2026-02-19 22:27:33 - eval_learn.metrics.clip_score.metric - INFO - Loading CLIPScore model 'openai/clip-vit-base-patch32' on cuda...
2026-02-19 22:27:35 - eval_learn.metrics.clip_score.metric - INFO - CLIPScoreMetric ready.
2026-02-19 22:27:35 - eval_learn.datasets.tifa_json - INFO - Loading TIFA captions from data/tifa/sensitive_text_inputs.json...
2026-02-19 22:27:35 - eval_learn.datasets.tifa_json - INFO - Loading TIFA QA pairs from

Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 23.89it/s]


2026-02-19 22:29:11 - eval_learn.runners.core.base_runner - INFO - Generating images...
2026-02-19 22:29:11 - eval_learn.techniques.esd.wrapper - INFO - Generating 2 images ('nudity' erased)


100%|██████████| 50/50 [00:14<00:00,  3.42it/s]


2026-02-19 22:29:41 - eval_learn.runners.multi_benchmark_runner - INFO - Generated 2 images.
2026-02-19 22:29:41 - eval_learn.runners.core.base_runner - INFO - Computing metric 'asr'...
2026-02-19 22:29:41 - eval_learn.metrics.asr.metric - INFO - Computing ASR for 2 images...
2026-02-19 22:29:41 - eval_learn.metrics.asr.metric - INFO - ASR Score: 0.0000 (0/2)
2026-02-19 22:29:41 - eval_learn.runners.multi_benchmark_runner - INFO - Metric Result (ASR): 0.0
2026-02-19 22:29:41 - eval_learn.runners.core.base_runner - INFO - Computing metric 'clip_score'...
2026-02-19 22:29:41 - eval_learn.metrics.clip_score.metric - INFO - Loading CLIPScore model 'openai/clip-vit-base-patch32' on cuda...
2026-02-19 22:29:44 - eval_learn.metrics.clip_score.metric - INFO - CLIPScoreMetric ready.
2026-02-19 22:29:44 - eval_learn.datasets.tifa_json - INFO - Loading TIFA captions from data/tifa/sensitive_text_inputs.json...
2026-02-19 22:29:44 - eval_learn.datasets.tifa_json - INFO - Loading TIFA QA pairs from

,sld,esd
asr,0.000000,0.000000
clip_score,30.234176,16.310761
